# PANDA Triple-Axis Spectrometer — McStas Simulations

Wavelenght:
$$\lambda=\frac{h}{p}$$
Wavevector:
$$k=\frac{2\pi}{\lambda}$$
Braggs-Law:
$$ 2d\sin(\Theta)=\Lambda$$

In [1]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import os


A Long time ago some smart people created a software to simulate neutrons, lets hope we can understand this 

First we write a simple source from the mccstas library in file:///C:/mcstas-3.6.5/lib/doc/manuals/mcdoc.html for some reoson this is at the very end

# The Source_gen Component

This routine is a neutron source (rectangular or circular), which aims at
a square target centered at the beam, Panda uses a rectangular source.



### Energy / Wavelength Distribution

The neutron energy/wavelength is distributed **uniformly in wavelength** between:

$$E_{min} = E_0 - \Delta E \quad \text{and} \quad E_{max} = E_0 + \Delta E$$

or equivalently:

$$\lambda_{min} = \lambda_0 - \Delta\lambda \quad \text{and} \quad \lambda_{max} = \lambda_0 + \Delta\lambda$$

The flux $I_1$ may be either arbitrary ($I_1 = 0$), or specified in:

$$\left[\frac{\text{n}}{\text{sr} \cdot \text{cm}^2 \cdot \text{Å} \cdot \text{s}}\right]$$

A Maxwellian spectrum may be selected by specifying source temperatures (up to 3).

### File Inputs

| Parameter | Description | Format |
|-----------|-------------|--------|
| `flux_file` | Flux vs. wavelength $[\lambda(\text{Å}),\ \phi(\text{n/s/cm}^2/\text{st/Å})]$ | 2-column free text |
| `xdiv_file` | Horizontal phase space $(x, \delta_x)$ distribution | Free text with `# xylimits:` header |
| `ydiv_file` | Vertical phase space $(y, \delta_y)$ distribution | Free text with `# xylimits:` header |

Phase space files require a comment line specifying axis limits:
```
# xylimits: pos_min pos_max div_min div_max
```

These files can be generated with `Monitor_nD`, e.g.:
```
Monitor_nD(options="auto lambda per cm2")
Monitor_nD(options="x hdiv, all auto")
Monitor_nD(options="y vdiv, all auto")
```

### Source Geometry

- **Circular**: defined by `radius`
- **Rectangular**: set non-zero `yheight` and `xwidth`
- **Thickness**: broadens the default zero time distribution $\Delta t = 0$

### Usage Examples
```c
Source_gen(radius=0.1, lambda0=2.36, dlambda=0.16, T1=20, I1=1e13,
           focus_xw=0.01, focus_yh=0.01)

Source_gen(yheight=0.1, xwidth=0.1, Emin=1, Emax=3, I1=1e13,
           verbose=1, focus_xw=0.01, focus_yh=0.01)

Parameters for Panda
           
FRM2 cold,20MW      T1=35.0, I1=9.38e12,T2=547.5,I2=2.23e12,T3=195.4,I3=1.26e13
```

# Panda rectangular neutron source 

```c
COMPONENT ColdSource = Source_gen(
   yheight  = 0.08,    xwidth   = 0.135,
   focus_xw = 0.11,    focus_yh = 0.1,
   dist     = 0.79045 + 0.39063,
   lambda0  = Lam,     dlambda  = dLam,
   T1=361.9, I1=7.22e12,
   T2=159.0, I2=6.74e12,
   T3=35.66, I3=6.435e12)
AT (0,0,0) RELATIVE origin
ROTATED (0,0,0) RELATIVE origin

## Arm Component

An arm does not actually do anything, it is just there to set
up a new coordinate system

```c
COMPONENT PrimaryBeam = Arm ()
  AT (0,0,0)  RELATIVE origin
  ROTATED (0,0,0) RELATIVE origin

## Guide Gravity

Neutron straight guide with gravity. Can be channeled and focusing. Waviness may be specified, as well as side chamfers (on substrate).

Geometry
The guide is defined by its entry and exit cross-sections and total length:
$$
w1×h1\xrightarrow{\quad l \quad} w2 \times h2​
$$
A long guide of length ll
l may be split into
nelements segments, between which chamfers and gaps are positioned to model a realistic assembly.

An important concept for this guide is the reflectivity, that describes the transit of light from a medium with otical density n to a medium with a lower optical density n''
$$
\Theta= 0.099 \cdot m \lambda(\text{Å})
$$

| Parameter  | Description                                              |
|------------|----------------------------------------------------------|
| `w1`       | Width at the guide entrance                              |
| `h1`       | Height at the guide entrance                             |
| `w2`       | Width at the guide exit                                  |
| `h2`       | Height at the guide exit                                 |
| `l`        | Length of the guide                                      |
| `R0`       | Low-angle reflectivity                                   |
| `Qc`       | Critical scattering vector                               |
| `alpha`    | Slope of reflectivity                                    |
| `W`        | Width of supermirror cut-off                             |
| `nslit`    | Number of vertical channels (≥ 1); `nslit-1` dividing walls |
| `mleft`    | *m*-value for left vertical mirror                       |
| `mright`   | *m*-value for right vertical mirror                      |
| `mtop`     | *m*-value for top horizontal mirror                      |
| `mbottom`  | *m*-value for bottom horizontal mirror                   |



## Panda Neutronen Guide (Neutronen Leiter)

```c
COMPONENT NL_SR2_1 = Guide_gravity(
    w1=0.105, h1=0.098, w2=0.105, h2=0.118, l=1.307,
    R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
    mleft=3.0, mright=3.0, mtop=3.0, mbottom=3.0)
  AT (0,0,1.1960) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam

COMPONENT NL_SR2_2a = Guide_gravity(w1=0.106, h1=0.119, w2=0.106, h2=0.134, l=0.997,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=3.0, mright=3.0, mtop=0, mbottom=0)

  WHEN (prim_shut_pos==2)
  AT (0,0,2.540) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam

COMPONENT NL_SR2_2c = Guide_gravity(w1=0.040, h1=0.119, w2=0.040, h2=0.134, l=0.997,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=2.0, mright=2.0, mtop=0, mbottom=0)
  WHEN (prim_shut_pos==3)
  AT (0,0,2.540) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam

/*####################################################################*/
/* beam plug 3rd part inner beam port plate                           */
/*####################################################################*/

COMPONENT NL_SR2_3 = Guide_gravity(w1=0.107, h1=0.135, w2=0.107, h2=0.138, l=0.180,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=2.0, mright=2.0, mtop=0, mbottom=0)
  AT (0,0,3.5460) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam
  ```

# The Al_window Component
Aluminium window in the beam

```c
COMPONENT SR2_Beamportwindow = Al_window(thickness = 0.003)
  AT (0,0,3.8100) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam
```

## The Collimator_linear Component

A simple analytical Soller collimator with triangular transmission.

Soller collimator with rectangular opening and specified length. The
transmission function is an average and does not utilize knowledge of the
actual neutron trajectory. A zero divergence disables collimation (the
component then acts as a double slit).

**Example:**
```c
Collimator_linear(xmin=-0.1, xmax=0.1, ymin=-0.1, ymax=0.1,
                  length=0.25, divergence=40, transmission=0.7)
```

| Parameter     | Description                                                                          |
|---------------|--------------------------------------------------------------------------------------|
| `xwidth`      | Width of slits                                                                       |
| `yheight`     | Height of slits                                                                      |
| `divergence`  | Horizontal divergence angle; calculated as arctan(*d* / `length`), *d* = blade spacing |
| `length`      | Distance between input and output slits                                              |
| `transmission`| Transmission of Soller (0 ≤ *t* ≤ 1)                                                |

```c
COMPONENT PANDA_ca1 = Collimator_linear(
	xwidth = 0.108, yheight = 0.160, divergence = ca1_div, length = 0.50, transmission = 1)
  WHEN (ca1_div < 80)
  AT (0,0,0.1000) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob
```


# The Filter_gen Component

This component may either set the flux or change it (filter-like), using an external data file.

This component changes the neutron flux (weight) to match a reference table from a file.
Typically you may set the neutron flux (source-like), or multiply it using a transmission
table (filter-like). The component may be placed after a source to simulate a real source
from a reference table, or used as a filter (BeO) or as a window (Al). The behaviour is
specified via the `options` parameter, or inferred from the filename itself (see below).

If the thickness for the transmission data file *D* was *t₀*, and a different thickness
*t₁* is required, the resulting transmission is:

$$D^{t_1/t_0}$$

Use the `thickness` and `scaling` parameters for that purpose.

| Parameter    | Description                                                                                                          |
|--------------|----------------------------------------------------------------------------------------------------------------------|
| `filename`   | Name of the data file to read (first two columns used). Data *D* should be sorted in ascending order and monotonic. The file may contain `options` as comments (see below). |
| `thickness`  | Relative thickness applied as *D* = *D*^(`thickness`)                                                               |




```c
COMPONENT PANDA_sapphire = Filter_gen(
	filename = "Al2O3_sapphire.trm", thickness = 0.86)
  WHEN (sapphire==1)
  AT (0,0,0.9000) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob
```


## The Slit Component

A simple rectangular or circular slit. Specify either the radius (circular) or the
rectangular bounds. No transmission outside the slit is permitted.

**Examples:**
```c
Slit(xmin=-0.01, xmax=0.01, ymin=-0.01, ymax=0.01)
Slit(xwidth=0.02, yheight=0.02)
Slit(radius=0.01)
```

| Parameter | Description                                        |
|-----------|----------------------------------------------------|
| `xwidth`  | Width of slit; overrides `xmin`/`xmax` if unset   |
| `yheight` | Height of slit; overrides `ymin`/`ymax` if unset  |

```c
/* beam channel until virtual source                 */

COMPONENT sk1_in = Slit(
	xwidth = 0.11, yheight = 0.15)
  AT (0,0,1.0100) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

COMPONENT sk1_out = Slit(
	xwidth = 0.09, yheight = 0.180)
  AT (0,0,2.0300) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

/*####################################################################*/
/* PANDA virtual source slit                                          */
/*####################################################################*/


COMPONENT PANDA_ms1 = Slit(
	xwidth = ms1, yheight = 0.180)
  AT (0,0,2.1800) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

/* beam channel from virtual source to mono drum                       */

COMPONENT sk2_in = Slit(
	xwidth = 0.050, yheight = 0.180)
  AT (0,0,2.4760) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

COMPONENT sk2_out = Slit(
	xwidth = 0.070, yheight = 0.200)
  AT (0,0,2.9360) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob


## The PSD_monitor Component

An (n times m) pixel PSD monitor. This component may also be used as a beam
detector.

Example: PSD_monitor(xmin=-0.1, xmax=0.1, ymin=-0.1, ymax=0.1, nx=90, ny=90, filename="Output.psd")

| Parameter        | Description                                      |
|------------------|--------------------------------------------------|
| `xmin`           | Lower *x* bound of detector opening             |
| `xmax`           | Upper *x* bound of detector opening             |
| `ymin`           | Lower *y* bound of detector opening             |
| `ymax`           | Upper *y* bound of detector opening             |
| `nx`             | Number of pixel columns                          |
| `ny`             | Number of pixel rows                             |
| `restore_neutron`| If set, the monitor does not affect neutron state|


```c 

COMPONENT PSD_PrimBeam = PSD_monitor(xmin=-0.04, xmax=0.04, ymin=-0.10, ymax=0.10,
                                          nx=80, ny=200, filename="PSD_PrimBeam.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS
  ```

## The L_monitor Component

A square single monitor that measures the wavelength of incoming neutrons.

**Example:**
```c
L_monitor(xmin=-0.1, xmax=0.1, ymin=-0.1, ymax=0.1,
          nL=20, filename="Output.L", Lmin=2, Lmax=10)
```

| Parameter        | Description                                               |
|------------------|-----------------------------------------------------------|
| `xmin`           | Lower *x* bound of detector opening                      |
| `xmax`           | Upper *x* bound of detector opening                      |
| `ymin`           | Lower *y* bound of detector opening                      |
| `ymax`           | Upper *y* bound of detector opening                      |
| `nL`             | Number of wavelength channels                             |
| `filename`       | Name of file in which to store the detector image         |
| `restore_neutron`| If set, the monitor does not affect neutron state         |
| `Lmin`           | Minimum wavelength to detect                              |
| `Lmax`           | Maximum wavelength to detect                              |


```c

COMPONENT LAM_PrimBeam = L_monitor(xmin=-0.04, xmax=0.04, ymin=-0.15, ymax=0.15,
                                        nL=100, filename="LAM_PrimBeam.L", restore_neutron=1,
                                        Lmin=MINLam, Lmax=MAXLam)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOUS
```

COMPONENT DIV_PrimBeam = Divergence_monitor(xmin=-0.04, xmax=0.04, ymin=-0.15, ymax=0.15,
					maxdiv_h = 2, maxdiv_v = 2, nh = 40, nv = 40, filename="DIV_PrimBeam.dat", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOUS


```

## The Divergence_monitor Component

A divergence-sensitive monitor. Counts are distributed across *n* × *m* pixels.

**Example:**
```c
Divergence_monitor(nh=20, nv=20, filename="Output.pos",
                   xmin=-0.1, xmax=0.1, ymin=-0.1, ymax=0.1,
                   maxdiv_h=2, maxdiv_v=2)
```

| Parameter        | Description                                               |
|------------------|-----------------------------------------------------------|
| `xmin`           | Lower *x* bound of detector opening                      |
| `xmax`           | Upper *x* bound of detector opening                      |
| `ymin`           | Lower *y* bound of detector opening                      |
| `ymax`           | Upper *y* bound of detector opening                      |
| `maxdiv_h`       | Maximum horizontal divergence detected                    |
| `maxdiv_v`       | Maximum vertical divergence detected                      |
| `nh`             | Number of horizontal pixel columns                        |
| `nv`             | Number of vertical pixel rows                             |
| `filename`       | Name of file in which to store the detector image         |
| `restore_neutron`| If set, the monitor does not affect neutron state         |


```c

COMPONENT DIV_PrimBeam = Divergence_monitor(xmin=-0.04, xmax=0.04, ymin=-0.15, ymax=0.15,
					maxdiv_h = 2, maxdiv_v = 2, nh = 40, nv = 40, filename="DIV_PrimBeam.dat", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOUS


```

## The Monochromator_curved Component

A double-bent, infinitely thin mosaic crystal, useful as a monochromator or analyzer.
Uses a small-mosaicity approximation and accounts for higher-order scattering. The mosaic
is anisotropic Gaussian with different FWHMs in the *y* and *z* directions. The scattering
vector is perpendicular to the surface. For an unrotated component, the crystal plane lies
in the *y*-*z* plane (i.e. parallel to the beam). The component works in reflection but also
transmits the non-diffracted beam. Reflectivity and transmission files may be used.

The slabs are positioned in the vertical plane (not on a cylinder or sphere) and rotated
according to the curvature radius. When curvatures are set to 0, the monochromator is flat.
The curvature approximations for parallel beam focusing to distance *L*, with monochromator
rotation angle *A1*, are:
```c
RV = 2*L*sin(DEG2RAD*A1);
RH = 2*L/sin(DEG2RAD*A1);
```

**Example:**
```c
Monochromator_curved(zwidth=0.01, yheight=0.01, gap=0.0005,
                     NH=11, NV=11, mosaich=30.0, mosaicv=30.0,
                     r0=0.7, Q=1.8734)
```

| Parameter  | Description                                                                 |
|------------|-----------------------------------------------------------------------------|
| `zwidth`   | Horizontal width of an individual slab                                      |
| `yheight`  | Vertical height of an individual slab                                       |
| `gap`      | Typical gap between adjacent slabs                                          |
| `NH`       | Number of slabs horizontally                                                |
| `NV`       | Number of slabs vertically                                                  |
| `mosaich`  | Horizontal mosaic FWHM                                                      |
| `mosaicv`  | Vertical mosaic FWHM                                                        |
| `mosaic`   | Convenience setter: assigns `mosaich` = `mosaicv`                          |
| `RV`       | Radius of vertical focusing; flat when 0                                    |
| `RH`       | Radius of horizontal focusing; flat when 0                                  |
| `Q`        | Scattering vector                                                            |
| `reflect`  | Reflectivity filename — two-column text file \[*k*, *R*\]                  |


```c


/*####################################################################*/
/* PANDA monochromator                                                */
/*####################################################################*/

COMPONENT PANDA_mth = Arm ()
  AT (0,0,5.000)  RELATIVE SR2_eob
  ROTATED (0,180-a1,0) RELATIVE SR2_eob


COMPONENT a_mono = Arm ()
  AT (mty,0,mtx)  RELATIVE PANDA_mth
  ROTATED (mgy,0,0) RELATIVE PANDA_mth

SPLIT 20 COMPONENT PG002_mono=Monochromator_curved(zwidth=0.02,yheight=0.018,gap=0.002,NH=11,NV=11,
                             mosaic=20,RV=mfv,RH=mfh,Q=1.873,reflect="HOPG.rfl")
  AT (0,0,0) RELATIVE a_mono

COMPONENT PANDA_mtt = Arm ()
  AT (0,0,0)  RELATIVE PANDA_mth
  ROTATED (0,-a2,0) RELATIVE PrimaryBeam


COMPONENT sk3_in = Slit (	xwidth = 0.136, yheight = 0.174)
  AT (0,0,0.2660) RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt


COMPONENT PANDA_ca2 = Collimator_linear(
	xwidth = 0.040, yheight = 0.120, divergence = ca2_div, length = 0.20)
  WHEN (ca2_div < 61)
  AT (0,0,0.9000) RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

COMPONENT sk3_out = Slit (	xwidth = 0.06, yheight = 0.10)
  AT (0,0,1.1500) RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

COMPONENT PSD_mon1 = PSD_monitor(xmin=-0.04, xmax=0.04, ymin=-0.06, ymax=0.06,
                                          nx=80, ny=120, filename="PSD_mon1.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT LAM_mon1 = L_monitor(xmin=-0.04, xmax=0.04, ymin=-0.06, ymax=0.06,
                                        nL=100, filename="LAM_mon1.L", restore_neutron=1,
                                        Lmin=MINLam, Lmax=MAXLam)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOUS

COMPONENT PANDA_ss1 = Slit(
	xwidth = ss1_width, yheight = ss1_height)

  AT (0,0,1.7000) RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt


```

```c
/*####################################################################*/
/* PANDA sample table                                                 */
/*####################################################################*/

COMPONENT PSD_samplepos = PSD_monitor(xmin=-0.05, xmax=0.05, ymin=-0.1, ymax=0.1,
                                          nx=100, ny=200, filename="PSD_samplepos.psd", restore_neutron=1)
  AT (0,0,2.1000)  RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

COMPONENT Mon_samplepos2 = Monitor(xwidth = 0.01, yheight= 0.02, restore_neutron=1)
  AT (0,0,2.1000)  RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

COMPONENT PANDA_sth = Arm ()
  AT (0,0,2.1000)  RELATIVE PANDA_mtt
  ROTATED (0,a3,0) RELATIVE PANDA_mtt

COMPONENT sample = Arm ()
  AT (sty,stz,stx)  RELATIVE PANDA_sth
  ROTATED (sgy,0,sgx) RELATIVE PANDA_sth
```

## The Incoherent Component

```c

/*####################################################################*/
/* put sample here                                                    */
/*####################################################################*/
/* incoherent sample to test energy resolution */
COMPONENT PANDA_sample = Incoherent(radius=0.0125, yheight=0.04, focus_r=0.05, pack=1, target_index=3)
  AT (0,0,0)  RELATIVE sample
  ROTATED (0,0,0) RELATIVE sample



/* Pb crystal cube with 10mm oriented (200,020) sth = stt/2  = 54.95 
COMPONENT PANDA_cryst = 
   Single_crystal(xwidth = 0.01, yheight = 0.01, zdepth = 0.01,
     mosaic = 30, reflections = "../data/Pb.lau",
     ax =  4.95, ay = 0.00, az = 0.00,
     bx =  0.00, by = 4.95, bz = 0.00,
     cx =  0.00, cy = 0.00, cz = 4.95 )
  AT (0,0,0)  RELATIVE sample
  ROTATED (0,0,45) RELATIVE sample
*/
/* finds 0,0,2  with Q= 2.5386 and kf = 1.55 at sth = 144.95 */
/* finds 1,1,1  with Q= 2.1985 and kf = 1.55 at sth =   9.91 */



/* analyser arm */

COMPONENT PANDA_stt = Arm ()
  AT (0,0,0)  RELATIVE PANDA_sth
  ROTATED (0,a4,0) RELATIVE PANDA_mtt


COMPONENT PSD_scattered = PSD_monitor(xmin=-0.1, xmax=0.1, ymin=-0.1, ymax=0.1,
                      nx=100, ny=100, filename="scattered.psd", restore_neutron=1)
  AT (0,0,0.2) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS


COMPONENT PANDA_ss2 = Slit(
	xwidth = ss2_width, yheight = ss2_height)

  AT (0,0,0.4000) RELATIVE PANDA_stt
  ROTATED (0,0,0) RELATIVE PANDA_stt
```





```c

/*####################################################################*/
/* PANDA analyser                                                     */
/*####################################################################*/

COMPONENT sk4_in = Slit (xwidth = 0.04, yheight = 0.12)
  AT (0,0,lsa-0.600) RELATIVE PANDA_stt
  ROTATED (0,0,0) RELATIVE PANDA_stt

COMPONENT PANDA_ca3 = Collimator_linear(
	xwidth = 0.040, yheight = 0.140, divergence = ca3_div, length = 0.2)
  WHEN (ca3_div < 61)
  AT (0,0,lsa-0.550) RELATIVE PANDA_stt
  ROTATED (0,0,0) RELATIVE PANDA_stt

COMPONENT sk4_out = Slit (	xwidth = 0.06, yheight = 0.1)
  AT (0,0,lsa-0.200) RELATIVE PANDA_stt
  ROTATED (0,0,0) RELATIVE PANDA_stt

/* placee monitor mon2 */
COMPONENT PSD_mon2 = PSD_monitor(xmin=-0.05, xmax=0.05, ymin=-0.06, ymax=0.06,
                      nx=100, ny=120, filename="PSD_mon2.psd", restore_neutron=1)
  AT (0,0,0.1) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT PANDA_ath = Arm ()
  AT (0,0,lsa)  RELATIVE PANDA_stt
  ROTATED (0,180-a5,0) RELATIVE PANDA_stt

COMPONENT analyser = Arm ()
  AT (aty,0,0)  RELATIVE PANDA_ath
  ROTATED (agy,0,0) RELATIVE PANDA_ath

/* PG002 analyser */ 

SPLIT 20 COMPONENT PG002_ana=Monochromator_curved(zwidth=0.013,yheight=0.025,gap=0.003,NH=13,NV=6,
                             mosaic=20,RV=afv,RH=afh,Q=1.873,reflect="HOPG.rfl")
  AT (0,0,0) RELATIVE analyser

/* detector arm */

COMPONENT PANDA_att = Arm ()
  AT (0,0,0)  RELATIVE PANDA_ath
  ROTATED (0,-a6,0) RELATIVE PANDA_stt
```

```c
/*####################################################################*/
/* PANDA detector                                                     */
/*####################################################################*/

COMPONENT sk5_in = Slit (	xwidth = 0.04, yheight = 0.12)
  AT (0,0,0.4000) RELATIVE PANDA_att
  ROTATED (0,0,0) RELATIVE PANDA_att

COMPONENT PANDA_ca4 = Collimator_linear(
	xwidth = 0.040, yheight = 0.140, divergence = ca4_div, length = 0.2)
  WHEN (ca4_div < 80)
  AT (0,0,0.4100) RELATIVE PANDA_att
  ROTATED (0,0,0) RELATIVE PANDA_att

COMPONENT sk5_out = Slit (	xwidth = 0.03, yheight = 0.12)
  AT (0,0,lad-0.1) RELATIVE PANDA_att
  ROTATED (0,0,0) RELATIVE PANDA_att


/* place for detector tube */

COMPONENT a_detector = Arm ()
  AT (0,0,lad)    RELATIVE PANDA_att
  ROTATED (0,0,0) RELATIVE PANDA_att

COMPONENT PSD_det = PSD_monitor(xmin=-0.03, xmax=0.03, ymin=-0.10, ymax=0.10,
                                          nx=60, ny=200, filename="PSD_det.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT PSD_det1 = PSD_monitor(xmin=-0.0125, xmax=0.0125, ymin=-0.05, ymax=0.05,
                                          nx=25, ny=100, filename="PSD_det1.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT PSD_det2 = PSD_monitor(xmin=-0.025, xmax=0.025, ymin=-0.05, ymax=0.05,
                                          nx=50, ny=100, filename="PSD_det2.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS



FINALLY
%{
%}
/* The END token marks the instrument definition end */
END
```

/*******************************************************************************
* Instrument: template_simple
*
* %I
* Written by: your name (email)
* Date: current date
* Origin: your institution
* %INSTRUMENT_SITE: Templates
*
* instrument short description
*
* %D
* instrument description
*
* Example: parameters=values
*
* %P
* Par1: [unit] Parameter1 description
*
* %L
* <reference/HTML link>
*
* %E
*******************************************************************************/
DEFINE INSTRUMENT template_simple(Par1=1)

DECLARE
%{
%}

USERVARS
%{
%}

INITIALIZE
%{
%}

TRACE

COMPONENT origin = Progress_bar()
AT (0, 0, 0) RELATIVE ABSOLUTE

// insert components here (e.g. Insert -> Source -> ...)

COMPONENT arm_source = Arm( )
AT (0, 0, 0) RELATIVE origin

COMPONENT source_simple = Source_simple( 
    radius=0.02, 
    dist=3, 
    focus_xw=0.01, 
    focus_yh=0.01,
    lambda0=6, 
    dlambda=0.05, 
    flux=1E8)
AT (0, 0, 0) RELATIVE PREVIOUS

COMPONENT coll_enter = Slit(
    radius=0.005)
AT (0, 0, 3) RELATIVE PREVIOUS

COMPONENT coll_exit = Slit( 
    radius=0.005)
AT (0, 0, 3) RELATIVE PREVIOUS

COMPONENT sansspheres = SANSSpheres( 
    R=100.0, 
    xwidth=0.01, 
    yheight=0.01, 
    zdepth=0.005, 
    SampleToDetectorDistance=5.8, 
    DetectorRadius=1)
AT (0, 0, 0.2) RELATIVE PREVIOUS

COMPONENT beamstop = Beamstop( 
    radius=0.02)
AT (0, 0, 2.8) RELATIVE PREVIOUS

COMPONENT psd_monitor = PSD_monitor( 
    nx=90, 
    ny=90, 
    filename="PSD_mon.dat",
    xwidth=1.2, 
    yheight=1.2)
AT (0, 0, 3) RELATIVE PREVIOUS

COMPONENT l_monitor = L_monitor( 
    nL=1000, 
    filename="L_mon.dat", 
    xwidth=1.2, 
    yheight=1.2, 
    Lmin=5.5, 
    Lmax=6.5, 
    restore_neutron=0)
AT (0, 0, 1E-9) RELATIVE PREVIOUS

COMPONENT psd_monitor_rad = PSD_monitor_rad( 
    nr=100, 
    filename="PSD_rad.dat", 
    filename_av="PSD_rad_av.dat", 
    rmax=0.6)
AT (0, 0, 1E-9) RELATIVE PREVIOUS

COMPONENT sansqmonitor = SANSQMonitor( 
    RFilename="RDetector", 
    qFilename="QDetector", 
    NumberOfBins=100, 
    restore_neutron=0, 
    RadiusDetector=0.6, 
    DistanceFromSample=5.8)
AT (0, 0, 1E-9) RELATIVE PREVIOUS

FINALLY
%{
%}



FINALLY
%{
%}

END


COMPONENT NL_SR2_1 = Guide_gravity(w1=0.105, h1=0.098, w2=0.105, h2=0.118, l=1.307,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=3.0, mright=3.0, mtop=3.0, mbottom=3.0)
  AT (0,0,1.1960) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam


COMPONENT PSD_PrimBeam = PSD_monitor(xmin=-0.04, xmax=0.04, ymin=-0.10, ymax=0.10,
                                          nx=80, ny=200, filename="PSD_PrimBeam.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS


COMPONENT LAM_PrimBeam = L_monitor(xmin=-0.04, xmax=0.04, ymin=-0.15, ymax=0.15,
                                        nL=100, filename="LAM_PrimBeam.L", restore_neutron=1,
                                        Lmin=MINLam, Lmax=MAXLam)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOUS



COMPONENT DIV_PrimBeam = Divergence_monitor(xmin=-0.04, xmax=0.04, ymin=-0.15, ymax=0.15,
					maxdiv_h = 2, maxdiv_v = 2, nh = 40, nv = 40, filename="DIV_PrimBeam.dat", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOU

In [ ]:
/*******************************************************************************
*         McStas instrument definition URL=http://www.mcstas.org
*
* Instrument: PANDA
*
* %Identification
* Written by: Peter Link (peter.link@frm2.tum.de)
* Date: 26.08.2019
* Origin: Forschungs-Neutronenquelle Heinz Maier-Leibnitz (FRM II)
* Release: McStas
* Version: 2.6.1
* %INSTRUMENT_SITE: FRM-II
*
* Instrument short description
*
* %PANDA - Cold triple axis spectrometer
* PANDA is the cold triple spectrometer at the beamtube SR2 of FRM II.
* Monochromator, Analyser: PG002
*
* Example: mcrun panda.instr <parameters=values>
*
* %Parameters
* Par1: [unit] Parameter1 description
* prim_shut_pos: [] select primary shutter opening 1,2 or 3       
* sapphire: [] select sapphire filter 1 = in , else out
* ca1: [min] alpha 1 collimation 20,40,60, 80 = open
* ca2: [min] alpha 2 collimation 15,40,60, 80 = open    
* ca3: [min] alpha 3 collimation 15,40,60, 80 = open 
* ca4: [min] alpha 4 collimation 15,40,60, 80 = open
* ms1: [m] width of virtual source slit
* lms: [m] distance monochromator center to sample center
* lsa: [m] distance sample center to analyser center
* lad: [m] distance analyser center to detector
* ki: [A-1] incident wavevector, if 0 not used
* kf: [A-1] final wavevector, if 0 not used
* dE: [meV] energy transfer  (meV)
* Q: [A-1] momentum transfer in (AA-1)
* mth: [deg] theta monochromator
* mtt: [deg] 2theta monochromator
* mtx: [mm] x translation monochromator
* mty: [mm] y translation monochromator
* mgy: [deg] y gonio monochromator
* mfh: [m] horizontal focus monochromator
* mfv: [m] vertical focus monochromator
* scatsense: [0] scattering sense at sample
* sth: [deg] theta sample
* stt: [deg] 2theta sample
* stx: [m] x trans sample
* sty: [m] y trans sample
* stz: [m] z trans sample
* sgx: [deg] x gonio sample
* sgy: [deg] y gonio sample
* ss1_width: [m] slit before sample width
* ss1_height: [m] slit before sample height
* ss2_width: [m] slit  after sample width
* ss2_height: [m] slit after sample height
* ath: [deg] theta analyser
* att: [deg] 2theta analyser
* atx: [m] x trans analyser
* aty: [m]y trans analyser
* agy: [deg] y gonio analyser
* afh: [m] horizontal focus analyser
* afv: [m] vertical focus  analyser
*
* %Link 
* <a href="http://dx.doi.org/10.17815/jlsrf-1-35">MLZ instrument PANDA</a>
* 
*
* %End
*******************************************************************************/

/* Change name of instrument and input parameters with default values */
DEFINE INSTRUMENT PANDA(
	int prim_shut_pos=1,  /* 1,2,  or 3 primary shutter opening        */
	int sapphire = 1,     /* sapphire filter 1 = in , else out         */
	int ca1 = 80,         /* alfa 1 collimation 20,40,60, 80 = open    */
	int ca2 = 80,         /* alfa 2 collimation 15,40,60, 80 = open    */
	int ca3 = 80,         /* alfa 3 collimation 15,40,60, 80 = open    */
	int ca4 = 80,         /* alfa 4 collimation 15,40,60, 80 = open    */
	double ms1 = 0.04,    /* width of virtual source                   */
  double lms = 2.15,    /* distance mono to sample                   */
	double lsa = 1.05,    /* distance sample to ana                    */
	double lad = 0.95,    /* distance ana to detector                  */
  double ki = 0,        /* incident wavevector, if 0 not used        */
	double kf = 0,        /* final wavevector, if 0 not  used          */ 
	double dE = 0,        /* energy transfer  (meV)                    */
  double  Q = 0,        /* momentum transfer in (AA-1)               */
/* mono axis */
	double mth = 0,       /* theta monochromator 					             */
	double mtt = 0,       /* 2theta monochromator 					           */
	double mtx = 0,       /* x translation monochromator               */
	double mty = 0,       /* y translation monochromator               */
	double mgy = 0,       /* gonio  monochromator 					           */
	double mfh = 4.5,     /* horizontal focus monochromator            */
	double mfv = 2,       /* vertical focus monochromator		           */
/* sample table */
	int scatsense = -1,   /* scattering sense at sample                */
  double sth = 0,       /* theta sample       					             */
	double stt = 0,       /* 2theta sample        					           */
  double stx = 0,       /* x trans sample                            */
  double sty = 0,       /* y trans sample                            */
  double stz = 0,       /* z trans sample                            */
  double sgx = 0,       /* x gonio sample                            */
  double sgy = 0,       /* y gonio sample                            */
  double ss1_width = 0.02, /* ss1 width                                 */
  double ss1_height = 0.03,/* ss1 height                                */
  double ss2_width = 0.02,  /* ss2 width                                */
  double ss2_height = 0.04, /* ss2 height                               */
/* analyser table */
	double ath = 0,       /* theta analyser     					             */
	double att = 0,       /* 2theta analyser      					           */
  double atx = 0,       /* x trans analyser                          */
  double aty = 0,       /* y trans analyser                          */
  double agy = 0,       /* y gonio analyser                          */
  double afh = 1.7,
  double afv = 0.6  )   /* focus  analyser                           */

// sample info
//	double h = 0,       /* sample Miller index h 					             */
//	double k = 0,       /* sample Miller index k   					           */
//  double l = 0,       /* sample Miller index l   					           */
//	double a = 0,       /* sample cell a axis    					             */
//	double b = 0,       /* sample cell b axis     					           */
//  double c = 0,       /* sample cell c axis     					           */
//	double alfa = 0,    /* sample cell a angle   					             */
//	double beta = 0,    /* sample cell b angle    					           */
//  double gamma = 0,   /* sample cell c angle    					           */

/* The DECLARE section allows us to declare variables or  small      */
/* functions in C syntax. These may be used in the whole instrument. */
DECLARE
%{	
 double ca1_div,ca2_div,ca3_div,ca4_div;
 double a1,a2,a3,a4,a5,a6;
 double my_ki,my_kf,my_thetas;
 double Lam,dLam,MINLam,MAXLam;
 double d_PG002;
%}

/* The INITIALIZE section is executed when the simulation starts     */
/* (C code). You may use them as component parameter values.         */
INITIALIZE
%{
 Lam = 4.0;
 dLam = 0.1;
 
 d_PG002 = 3.35461;

 switch(ca1)
     {
        case 20 : ca1_div = 20;break; 
        case 40 : ca1_div = 40;break;
        case 60 : ca1_div = 60;break;
		default:  ca1_div = 120;}

 switch(ca2)
     {
        case 15 : ca2_div = 15;break; 
        case 40 : ca2_div = 40;break;
        case 60 : ca2_div = 60;break;
		default:  ca2_div = 120;}

 switch(ca3)
     {
        case 15 : ca3_div = 15;break; 
        case 40 : ca3_div = 40;break;
        case 60 : ca3_div = 60;break;
		default:  ca3_div = 120;}

 switch(ca4)
     {
        case 15 : ca4_div = 15;break; 
        case 40 : ca4_div = 40;break;
        case 60 : ca4_div = 60;break;
		default:  ca4_div = 120;}

/* input data decides the relevant params */
  a1=0;a2=0;a3=0;a4=0;a5=0;a6=0;
 if (kf) {
	 /* calculate default ath,att angles from kf */
	 my_kf = kf;
   if (dE){my_ki = sqrt(dE / 2.072 + kf*kf);}   // meV and A-1
   else {my_ki = kf;};
   Lam = 2*PI/my_ki;
   dLam = 0.03 * Lam;}
 else if (ki) {
	 /* calculate default mth,mtt angles from ki */
   my_ki = ki;
   if (dE){my_kf = sqrt( ki*ki - dE /2.072);}  // meV and A-1
   else { my_kf = ki;};                 // dE = 0 : ki = kf  
   Lam = 2*PI/my_ki;
   dLam = 0.03 * Lam;}

/* sample related parameters */


 if ((Q) && (my_ki) && (my_kf)) { // calculate stt from momentum transfer
   my_thetas = acos ( (my_ki*my_ki + my_kf*my_kf - Q*Q)/ 2/my_ki/my_kf); 
   a4 = my_thetas*RAD2DEG;}
 else {a4 = stt;};  
 a3 = sth;
 
 a1 = asin ( PI/d_PG002/my_ki)*RAD2DEG;
 a2 = 2*a1;
 a5 = asin ( PI/d_PG002/my_kf)*RAD2DEG;
 a6 = 2*a5;
 MINLam = Lam -dLam;
 MAXLam = Lam +dLam;

 printf("ki: %f A-1\n",my_ki);
 printf("kf: %f A-1\n",my_kf);
 printf("dE: %f meV\n",dE);
 printf("Lam: %f A\n",Lam);
 printf("dLam: %f A\n",dLam);
 
 printf("mth: %f deg\n",a1);
 printf("mtt: %f deg\n",a2);
 printf("sth: %f deg\n",a3);
 printf("stt: %f deg\n",a4);
 printf("ath: %f deg\n",a5);
 printf("att: %f deg\n",a6);
  

%}

/* Here comes the TRACE section, where the actual      */
/* instrument is defined as a sequence of components.  */
TRACE

/* The Arm() class component defines reference points and orientations  */
/* in 3D space. Every component instance must have a unique name. Here, */
/* Origin is used. This Arm() component is set to define the origin of  */
/* our global coordinate system (AT (0,0,0) ABSOLUTE). It may be used   */
/* for further RELATIVE reference, Other useful keywords are : ROTATED  */
/* EXTEND GROUP PREVIOUS. Also think about adding a neutron source !    */
/* Progress_bar is an Arm displaying simulation progress.               */


COMPONENT origin = Progress_bar()

  AT (0,0,0) ABSOLUTE

/* in the center of beam nose area = illuminating face   				*/

/*####################################################################*/
/* Source */
/*####################################################################*/



/* See drawing ZZD544-NLFM-00-001261 for the SR2 beam tube  and          */
/* drawing ZG-0821-020 for the beam tube nose                            */ 
/* the mc source is placed at the intersection point of the beam         */
/* direction with the cold source outer wall 							               */  

COMPONENT ColdSource = Source_gen(
   yheight =  0.08, xwidth = 0.135,
   focus_xw = 0.11, focus_yh = 0.1, dist = 0.79045+0.39063,
   lambda0 = Lam, dlambda = dLam,
   T1=361.9,I1=7.22e12,
   T2=159.0,I2=6.74e12,
   T3=35.66,I3=6.435e12)
  AT (0,0,0)  RELATIVE origin
  ROTATED (0,0,0) RELATIVE origin


/*####################################################################*/
/*Source monitors*/
/*####################################################################*/

COMPONENT PrimaryBeam = Arm ()
  AT (0,0,0)  RELATIVE origin
  ROTATED (0,0,0) RELATIVE origin

/*####################################################################*/
/* beam plug 1st part neutron guide with m = 3 coating                 */
/*####################################################################*/

COMPONENT NL_SR2_1 = Guide_gravity(w1=0.105, h1=0.098, w2=0.105, h2=0.118, l=1.307,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=3.0, mright=3.0, mtop=3.0, mbottom=3.0)
  AT (0,0,1.1960) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam

/*####################################################################*/
/* beam plug 2nd part revolver with three positions                   */
/*####################################################################*/

/* select primary shutter position 1,2 or 3                           */
COMPONENT NL_SR2_2a = Guide_gravity(w1=0.106, h1=0.119, w2=0.106, h2=0.134, l=0.997,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=3.0, mright=3.0, mtop=0, mbottom=0)
  WHEN (prim_shut_pos==1)
  AT (0,0,2.540) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam
  
COMPONENT NL_SR2_2b = Guide_gravity(w1=0.050, h1=0.099, w2=0.050, h2=0.107, l=0.997,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=0, mright=0, mtop=0, mbottom=0)
  WHEN (prim_shut_pos==2)
  AT (0,0,2.540) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam

COMPONENT NL_SR2_2c = Guide_gravity(w1=0.040, h1=0.119, w2=0.040, h2=0.134, l=0.997,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=2.0, mright=2.0, mtop=0, mbottom=0)
  WHEN (prim_shut_pos==3)
  AT (0,0,2.540) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam

/*####################################################################*/
/* beam plug 3rd part inner beam port plate                           */
/*####################################################################*/

COMPONENT NL_SR2_3 = Guide_gravity(w1=0.107, h1=0.135, w2=0.107, h2=0.138, l=0.180,
                                     R0=0.99, Qc=0.02174, alpha=3.0, W=0.001, nslit=1,
                                     mleft=2.0, mright=2.0, mtop=0, mbottom=0)
  AT (0,0,3.5460) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam

COMPONENT SR2_Beamportwindow = Al_window(thickness = 0.003)
  AT (0,0,3.8100) RELATIVE PrimaryBeam
  ROTATED (0,0,0) RELATIVE PrimaryBeam


/*####################################################################*/
/* end of SR2 beam tube = outer face of 3rd barrier                   */
/*####################################################################*/

COMPONENT SR2_eob = Arm()
  AT (0,0,0.0031) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS


COMPONENT PANDA_ca1 = Collimator_linear(
	xwidth = 0.108, yheight = 0.160, divergence = ca1_div, length = 0.50, transmission = 1)
  WHEN (ca1_div < 80)
  AT (0,0,0.1000) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

COMPONENT PANDA_sapphire = Filter_gen(
	filename = "Al2O3_sapphire.trm", thickness = 0.86)
  WHEN (sapphire==1)
  AT (0,0,0.9000) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

/* beam channel until virtual source                 */

COMPONENT sk1_in = Slit(
	xwidth = 0.11, yheight = 0.15)
  AT (0,0,1.0100) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

COMPONENT sk1_out = Slit(
	xwidth = 0.09, yheight = 0.180)
  AT (0,0,2.0300) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

/*####################################################################*/
/* PANDA virtual source slit                                          */
/*####################################################################*/


COMPONENT PANDA_ms1 = Slit(
	xwidth = ms1, yheight = 0.180)
  AT (0,0,2.1800) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

/* beam channel from virtual source to mono drum                       */

COMPONENT sk2_in = Slit(
	xwidth = 0.050, yheight = 0.180)
  AT (0,0,2.4760) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

COMPONENT sk2_out = Slit(
	xwidth = 0.070, yheight = 0.200)
  AT (0,0,2.9360) RELATIVE SR2_eob
  ROTATED (0,0,0) RELATIVE SR2_eob

COMPONENT PSD_PrimBeam = PSD_monitor(xmin=-0.04, xmax=0.04, ymin=-0.10, ymax=0.10,
                                          nx=80, ny=200, filename="PSD_PrimBeam.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT LAM_PrimBeam = L_monitor(xmin=-0.04, xmax=0.04, ymin=-0.15, ymax=0.15,
                                        nL=100, filename="LAM_PrimBeam.L", restore_neutron=1,
                                        Lmin=MINLam, Lmax=MAXLam)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOUS

COMPONENT DIV_PrimBeam = Divergence_monitor(xmin=-0.04, xmax=0.04, ymin=-0.15, ymax=0.15,
					maxdiv_h = 2, maxdiv_v = 2, nh = 40, nv = 40, filename="DIV_PrimBeam.dat", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOUS


/*####################################################################*/
/* PANDA monochromator                                                */
/*####################################################################*/

COMPONENT PANDA_mth = Arm ()
  AT (0,0,5.000)  RELATIVE SR2_eob
  ROTATED (0,180-a1,0) RELATIVE SR2_eob


COMPONENT a_mono = Arm ()
  AT (mty,0,mtx)  RELATIVE PANDA_mth
  ROTATED (mgy,0,0) RELATIVE PANDA_mth

SPLIT 20 COMPONENT PG002_mono=Monochromator_curved(zwidth=0.02,yheight=0.018,gap=0.002,NH=11,NV=11,
                             mosaic=20,RV=mfv,RH=mfh,Q=1.873,reflect="HOPG.rfl")
  AT (0,0,0) RELATIVE a_mono

COMPONENT PANDA_mtt = Arm ()
  AT (0,0,0)  RELATIVE PANDA_mth
  ROTATED (0,-a2,0) RELATIVE PrimaryBeam


COMPONENT sk3_in = Slit (	xwidth = 0.136, yheight = 0.174)
  AT (0,0,0.2660) RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt


COMPONENT PANDA_ca2 = Collimator_linear(
	xwidth = 0.040, yheight = 0.120, divergence = ca2_div, length = 0.20)
  WHEN (ca2_div < 61)
  AT (0,0,0.9000) RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

COMPONENT sk3_out = Slit (	xwidth = 0.06, yheight = 0.10)
  AT (0,0,1.1500) RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

COMPONENT PSD_mon1 = PSD_monitor(xmin=-0.04, xmax=0.04, ymin=-0.06, ymax=0.06,
                                          nx=80, ny=120, filename="PSD_mon1.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT LAM_mon1 = L_monitor(xmin=-0.04, xmax=0.04, ymin=-0.06, ymax=0.06,
                                        nL=100, filename="LAM_mon1.L", restore_neutron=1,
                                        Lmin=MINLam, Lmax=MAXLam)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0)  RELATIVE PREVIOUS

COMPONENT PANDA_ss1 = Slit(
	xwidth = ss1_width, yheight = ss1_height)

  AT (0,0,1.7000) RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

/*####################################################################*/
/* PANDA sample table                                                 */
/*####################################################################*/

COMPONENT PSD_samplepos = PSD_monitor(xmin=-0.05, xmax=0.05, ymin=-0.1, ymax=0.1,
                                          nx=100, ny=200, filename="PSD_samplepos.psd", restore_neutron=1)
  AT (0,0,2.1000)  RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

COMPONENT Mon_samplepos2 = Monitor(xwidth = 0.01, yheight= 0.02, restore_neutron=1)
  AT (0,0,2.1000)  RELATIVE PANDA_mtt
  ROTATED (0,0,0) RELATIVE PANDA_mtt

COMPONENT PANDA_sth = Arm ()
  AT (0,0,2.1000)  RELATIVE PANDA_mtt
  ROTATED (0,a3,0) RELATIVE PANDA_mtt

COMPONENT sample = Arm ()
  AT (sty,stz,stx)  RELATIVE PANDA_sth
  ROTATED (sgy,0,sgx) RELATIVE PANDA_sth


/*####################################################################*/
/* put sample here                                                    */
/*####################################################################*/
/* incoherent sample to test energy resolution */
COMPONENT PANDA_sample = Incoherent(radius=0.0125, yheight=0.04, focus_r=0.05, pack=1, target_index=3)
  AT (0,0,0)  RELATIVE sample
  ROTATED (0,0,0) RELATIVE sample



/* Pb crystal cube with 10mm oriented (200,020) sth = stt/2  = 54.95 
COMPONENT PANDA_cryst = 
   Single_crystal(xwidth = 0.01, yheight = 0.01, zdepth = 0.01,
     mosaic = 30, reflections = "../data/Pb.lau",
     ax =  4.95, ay = 0.00, az = 0.00,
     bx =  0.00, by = 4.95, bz = 0.00,
     cx =  0.00, cy = 0.00, cz = 4.95 )
  AT (0,0,0)  RELATIVE sample
  ROTATED (0,0,45) RELATIVE sample
*/
/* finds 0,0,2  with Q= 2.5386 and kf = 1.55 at sth = 144.95 */
/* finds 1,1,1  with Q= 2.1985 and kf = 1.55 at sth =   9.91 */



/* analyser arm */

COMPONENT PANDA_stt = Arm ()
  AT (0,0,0)  RELATIVE PANDA_sth
  ROTATED (0,a4,0) RELATIVE PANDA_mtt


COMPONENT PSD_scattered = PSD_monitor(xmin=-0.1, xmax=0.1, ymin=-0.1, ymax=0.1,
                      nx=100, ny=100, filename="scattered.psd", restore_neutron=1)
  AT (0,0,0.2) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS


COMPONENT PANDA_ss2 = Slit(
	xwidth = ss2_width, yheight = ss2_height)

  AT (0,0,0.4000) RELATIVE PANDA_stt
  ROTATED (0,0,0) RELATIVE PANDA_stt






/*####################################################################*/
/* PANDA analyser                                                     */
/*####################################################################*/

COMPONENT sk4_in = Slit (xwidth = 0.04, yheight = 0.12)
  AT (0,0,lsa-0.600) RELATIVE PANDA_stt
  ROTATED (0,0,0) RELATIVE PANDA_stt

COMPONENT PANDA_ca3 = Collimator_linear(
	xwidth = 0.040, yheight = 0.140, divergence = ca3_div, length = 0.2)
  WHEN (ca3_div < 61)
  AT (0,0,lsa-0.550) RELATIVE PANDA_stt
  ROTATED (0,0,0) RELATIVE PANDA_stt

COMPONENT sk4_out = Slit (	xwidth = 0.06, yheight = 0.1)
  AT (0,0,lsa-0.200) RELATIVE PANDA_stt
  ROTATED (0,0,0) RELATIVE PANDA_stt

/* placee monitor mon2 */
COMPONENT PSD_mon2 = PSD_monitor(xmin=-0.05, xmax=0.05, ymin=-0.06, ymax=0.06,
                      nx=100, ny=120, filename="PSD_mon2.psd", restore_neutron=1)
  AT (0,0,0.1) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT PANDA_ath = Arm ()
  AT (0,0,lsa)  RELATIVE PANDA_stt
  ROTATED (0,180-a5,0) RELATIVE PANDA_stt

COMPONENT analyser = Arm ()
  AT (aty,0,0)  RELATIVE PANDA_ath
  ROTATED (agy,0,0) RELATIVE PANDA_ath

/* PG002 analyser */ 

SPLIT 20 COMPONENT PG002_ana=Monochromator_curved(zwidth=0.013,yheight=0.025,gap=0.003,NH=13,NV=6,
                             mosaic=20,RV=afv,RH=afh,Q=1.873,reflect="HOPG.rfl")
  AT (0,0,0) RELATIVE analyser

/* detector arm */

COMPONENT PANDA_att = Arm ()
  AT (0,0,0)  RELATIVE PANDA_ath
  ROTATED (0,-a6,0) RELATIVE PANDA_stt

/*####################################################################*/
/* PANDA detector                                                     */
/*####################################################################*/

COMPONENT sk5_in = Slit (	xwidth = 0.04, yheight = 0.12)
  AT (0,0,0.4000) RELATIVE PANDA_att
  ROTATED (0,0,0) RELATIVE PANDA_att

COMPONENT PANDA_ca4 = Collimator_linear(
	xwidth = 0.040, yheight = 0.140, divergence = ca4_div, length = 0.2)
  WHEN (ca4_div < 80)
  AT (0,0,0.4100) RELATIVE PANDA_att
  ROTATED (0,0,0) RELATIVE PANDA_att

COMPONENT sk5_out = Slit (	xwidth = 0.03, yheight = 0.12)
  AT (0,0,lad-0.1) RELATIVE PANDA_att
  ROTATED (0,0,0) RELATIVE PANDA_att


/* place for detector tube */

COMPONENT a_detector = Arm ()
  AT (0,0,lad)    RELATIVE PANDA_att
  ROTATED (0,0,0) RELATIVE PANDA_att

COMPONENT PSD_det = PSD_monitor(xmin=-0.03, xmax=0.03, ymin=-0.10, ymax=0.10,
                                          nx=60, ny=200, filename="PSD_det.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT PSD_det1 = PSD_monitor(xmin=-0.0125, xmax=0.0125, ymin=-0.05, ymax=0.05,
                                          nx=25, ny=100, filename="PSD_det1.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS

COMPONENT PSD_det2 = PSD_monitor(xmin=-0.025, xmax=0.025, ymin=-0.05, ymax=0.05,
                                          nx=50, ny=100, filename="PSD_det2.psd", restore_neutron=1)
  AT (0,0,0.0001) RELATIVE PREVIOUS
  ROTATED (0,0,0) RELATIVE PREVIOUS



FINALLY
%{
%}
/* The END token marks the instrument definition end */
END































---- Found 1 places in McStas output with keyword 'error'. 

ERROR: Compilation failed for simple_source.c using compiler cl.exe

----------------------------------------------------------------------

INFO: Using directory: "c:\Users\Sierra_Manrique\Desktop\simple_source"
INFO: Regenerating c-file: simple_source.c
CFLAGS=

-----------------------------------------------------------

Generating single GPU kernel or single CPU section layout: 

-----------------------------------------------------------

Generating GPU/CPU -DFUNNEL layout:

-----------------------------------------------------------
INFO: Recompiling: simple_source.exe
'cl.exe' is not recognized as an internal or external command,
operable program or batch file.
INFO: call to cl.exe failed with Command 'cl.exe simple_source.c /DWIN32 /D_WINDOWS /Zi /O2 /Ob1 /DNDEBUG /IC:/Users/Sierra_Manrique/AppData/Roaming/mamba/envs/mcstas/Library/include -DNDEBUG -D_POSIX_SOURCE /link /LIBPATH:C:/Users/Sierra_Manrique/AppData/Roamin

gcc: 
cl: 
mcstas: C:\Users\Sierra_Manrique\AppData\Roaming\mamba\envs\mcstas\bin\mcstas.exe



3.6.8




Instrument file written!



INFO: No output directory specified (--dir)
INFO: Using directory: "simple_source_20260330_181231"
INFO: Using existing c-file: simple_source.c
INFO: Recompiling: simple_source.exe
'cl.exe' is not recognized as an internal or external command,
operable program or batch file.
INFO: call to cl.exe failed with Command 'cl.exe simple_source.c /DWIN32 /D_WINDOWS /Zi /O2 /Ob1 /DNDEBUG /IC:/Users/Sierra_Manrique/AppData/Roaming/mamba/envs/mcstas/Library/include -DNDEBUG -D_POSIX_SOURCE /link /LIBPATH:C:/Users/Sierra_Manrique/AppData/Roaming/mamba/envs/mcstas/Library/lib' returned non-zero exit status 1.
ERROR: Compilation failed for simple_source.c using compiler cl.exe



❌ gcc: not found
❌ cc: not found
❌ cl: not found
❌ mingw32-gcc: not found


3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

# All requested packages already installed.


(mcstas) c:\Users\Sierra_Manrique\Desktop>SET DISTUTILS_USE_SDK=1 

(mcstas) c:\Users\Sierra_Manrique\Desktop>SET MSSdk=1 

(mcstas) c:\Users\Sierra_Manrique\Desktop>SET "VS_VERSION=17.0" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>SET "VS_MAJOR=17" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>SET "VS_YEAR=2022" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>set "MSYS2_ARG_CONV_EXCL=/AI;/AL;/OUT;/out" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>set "MSYS2_ENV_CONV_EXCL=CL" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>set "PY_VCRUNTIME_REDIST=\bin\vcruntime140.dll" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>set "CXX=cl.exe" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>set "CC=cl.exe" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>set "VSINSTALLDIR=" 

(mcstas) c:\Users\Sierra_Manrique\Desktop>set "NEWER_VS_WITH_OLDER_VC=0" 

(mcstas) 

In [19]:
result = subprocess.run("where gcc", shell=True, capture_output=True, text=True)
print(result.stdout)

C:\Users\Sierra_Manrique\AppData\Roaming\mamba\envs\mcstas\Library\mingw-w64\bin\gcc.exe



In [30]:
import subprocess
import os

# Tell McStas to use gcc
env = os.environ.copy()
env["CC"] = "gcc"
env["MCSTAS_CC"] = "gcc"

result = subprocess.run(
    "mcrun simple_source.instr -n 1000000",
    shell=True,
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
print(result.stderr)


INFO: No output directory specified (--dir)
INFO: Using directory: "simple_source_20260330_181405"
INFO: Using existing c-file: simple_source.c
INFO: Recompiling: simple_source.exe
'cl.exe' is not recognized as an internal or external command,
operable program or batch file.
INFO: call to cl.exe failed with Command 'cl.exe simple_source.c /DWIN32 /D_WINDOWS /Zi /O2 /Ob1 /DNDEBUG /IC:/Users/Sierra_Manrique/AppData/Roaming/mamba/envs/mcstas/Library/include -DNDEBUG -D_POSIX_SOURCE /link /LIBPATH:C:/Users/Sierra_Manrique/AppData/Roaming/mamba/envs/mcstas/Library/lib' returned non-zero exit status 1.
ERROR: Compilation failed for simple_source.c using compiler cl.exe



In [24]:
import subprocess
import os

vs_bin = r"C:\Program Files\Microsoft Visual Studio\2022\Enterprise\VC\Tools\MSVC\14.44.35207\bin\Hostx64\x64"
vs_common = r"C:\Program Files\Microsoft Visual Studio\2022\Enterprise\Common7\IDE"
vs_tools = r"C:\Program Files\Microsoft Visual Studio\2022\Enterprise\Common7\Tools"

env = os.environ.copy()
env["PATH"] = vs_bin + ";" + vs_common + ";" + vs_tools + ";" + env["PATH"]

# Verify cl.exe is now found
check = subprocess.run("where cl", shell=True, capture_output=True, text=True, env=env)
print("cl.exe found:", check.stdout)

# Now try mcrun
result = subprocess.run(
    "mcrun simple_source.instr -n 1000000",
    shell=True,
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
print(result.stderr)

cl.exe found: 

INFO: No output directory specified (--dir)
INFO: Using directory: "simple_source_20260330_180729"
INFO: Using existing c-file: simple_source.c
INFO: Recompiling: simple_source.exe
'cl.exe' is not recognized as an internal or external command,
operable program or batch file.
INFO: call to cl.exe failed with Command 'cl.exe simple_source.c /DWIN32 /D_WINDOWS /Zi /O2 /Ob1 /DNDEBUG /IC:/Users/Sierra_Manrique/AppData/Roaming/mamba/envs/mcstas/Library/include -DNDEBUG -D_POSIX_SOURCE /link /LIBPATH:C:/Users/Sierra_Manrique/AppData/Roaming/mamba/envs/mcstas/Library/lib' returned non-zero exit status 1.
ERROR: Compilation failed for simple_source.c using compiler cl.exe



In [1]:
import subprocess

result = subprocess.run("where cl", shell=True, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)


INFO: Could not find files for the given pattern(s).



In [1]:
where cl 


SyntaxError: invalid syntax (2437271644.py, line 1)